# 08 — Ensemble LightGBM + XGBoost

**Materia:** Laboratorio de Implementación II · Universidad Austral · Abril 2026

**Autores:** Roxana Alberti · Sandra Sschicchi · Fernando Paganini · Baltazar Villanueva · Paula Calviello · Rosana Martinez

---

## ¿Qué hace este notebook?

Combinamos las predicciones de **LightGBM** (gradient boosting con hojas) y **XGBoost** (gradient boosting con árboles) sobre las mismas 48 features del FE v4.

### ¿Por qué combinar LGB + XGB?
Aunque ambos son modelos de gradient boosting, difieren en:
- **Estrategia de crecimiento**: LightGBM crece por hoja (*leaf-wise*), XGBoost crece por nivel (*depth-wise*)
- **Regularización**: LightGBM usa L1/L2 sobre pesos de hojas; XGBoost penaliza el número de hojas y la magnitud de los pesos
- **Manejo de features**: LightGBM usa histogramas con binning exclusivo; XGBoost usa sorted splits exactos

Estas diferencias hacen que **cometan errores en casos distintos**, y al promediar sus probabilidades el error total se reduce — esto es la esencia del *ensemble*.

| Modelo | Kappa Test |
|---|---|
| LightGBM FE v4 + CV | 0.3867 |
| XGBoost FE v4 + CV | (este notebook) |
| **Ensemble LGB + XGB** | **(este notebook)** |

## Sección A: Imports y datos

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split, StratifiedKFold
import optuna
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_DIR = Path.cwd()
while not (BASE_DIR / 'input').exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent
print(f'BASE_DIR: {BASE_DIR}')

SEED = 42
train_raw = pd.read_csv(BASE_DIR / 'input/train/train.csv')
sent_df   = pd.read_csv(BASE_DIR / 'input/train_sentiment_features.csv')
meta_df   = pd.read_csv(BASE_DIR / 'input/train_metadata_features.csv')
train_raw['desc_length'] = train_raw['Description'].fillna('').apply(len)

df = (train_raw
      .merge(sent_df[['PetID','sentiment_score','sentiment_magnitude','n_sentences']], on='PetID', how='left')
      .merge(meta_df[['PetID','avg_label_score','n_labels','crop_confidence']], on='PetID', how='left')
      .fillna(0))

train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['AdoptionSpeed'])
print(f'Train: {len(train)} | Test: {len(test)}')


BASE_DIR: C:\Users\User\Desktop\MCD\Laboratorio de Implementacion II\GitHub\UA_MDM_Labo2


Train: 11994 | Test: 2999


## Sección B: Feature Engineering v4

In [2]:
def target_encode(train_df, test_df, col, target='AdoptionSpeed', smoothing=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['encoded'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return train_df[col].map(stats['encoded']).fillna(global_mean), test_df[col].map(stats['encoded']).fillna(global_mean)

def add_features_v4(df_):
    df_ = df_.copy()
    df_['HasPhoto']            = (df_['PhotoAmt'] > 0).astype(int)
    df_['HasVideo']            = (df_['VideoAmt'] > 0).astype(int)
    df_['IsFree']              = (df_['Fee'] == 0).astype(int)
    df_['AgeGroup']            = pd.cut(df_['Age'], bins=[-1,3,12,48,9999], labels=[0,1,2,3]).astype(int)
    df_['HealthScore']         = ((df_['Vaccinated']==1).astype(int) + (df_['Dewormed']==1).astype(int) + (df_['Sterilized']==1).astype(int))
    df_['IsPureBreed']         = (df_['Breed2'] == 0).astype(int)
    df_['PhotoPerAnimal']      = df_['PhotoAmt'] / df_['Quantity'].replace(0,1)
    df_['Age_x_PhotoAmt']      = df_['Age'] * df_['PhotoAmt']
    df_['IsPureBreed_x_Age']   = df_['IsPureBreed'] * df_['AgeGroup']
    df_['HealthScore_x_Photo'] = df_['HealthScore'] * df_['HasPhoto']
    df_['IsYoungAndFree']      = ((df_['AgeGroup'] <= 1) & (df_['IsFree'] == 1)).astype(int)
    df_['IsHealthyAndPhoto']   = ((df_['HealthScore'] == 3) & (df_['HasPhoto'] == 1)).astype(int)
    df_['FeePerAnimal']        = df_['Fee'] / df_['Quantity'].replace(0,1)
    return df_

def nlp_feats(df_):
    desc = df_['Description'].apply(lambda x: '' if (x == 0 or str(x).strip() == '') else str(x))
    df_['word_count']      = desc.apply(lambda x: len(x.split()))
    df_['unique_words']    = desc.apply(lambda x: len(set(x.lower().split())))
    df_['avg_word_len']    = desc.apply(lambda x: round(sum(len(w) for w in x.split()) / max(len(x.split()),1), 2))
    df_['uppercase_ratio'] = desc.apply(lambda x: round(sum(c.isupper() for c in x) / max(len(x),1), 4))
    df_['has_exclamation'] = desc.apply(lambda x: int('!' in x))
    return df_

train = train.copy(); test = test.copy()
train['Breed1_enc'], test['Breed1_enc'] = target_encode(train, test, 'Breed1')
train['State_enc'],  test['State_enc']  = target_encode(train, test, 'State')

rescuer_count = train.groupby('RescuerID').size().rename('rescuer_n_pets')
train['rescuer_n_pets'] = train['RescuerID'].map(rescuer_count).fillna(1)
test['rescuer_n_pets']  = test['RescuerID'].map(rescuer_count).fillna(1)

age_med_map = train.groupby(['Breed1','Type'])['Age'].median().to_dict()
global_age  = train['Age'].median()
for df_ in [train, test]:
    df_['age_median_bt'] = [age_med_map.get((b,t), global_age) for b,t in zip(df_['Breed1'], df_['Type'])]
    df_['age_rel_breed'] = df_['Age'] / (df_['age_median_bt'] + 1)

train = nlp_feats(train); test = nlp_feats(test)
train_fe = add_features_v4(train); test_fe = add_features_v4(test)

ALL_FEATURES = [
    'Type','Age','Breed1','Breed2','Gender','Color1','Color2','Color3',
    'MaturitySize','FurLength','Vaccinated','Dewormed','Sterilized',
    'Health','Quantity','Fee','State','VideoAmt','PhotoAmt',
    'HasPhoto','HasVideo','IsFree','AgeGroup','HealthScore','IsPureBreed','PhotoPerAnimal',
    'Age_x_PhotoAmt','IsPureBreed_x_Age','HealthScore_x_Photo','IsYoungAndFree','IsHealthyAndPhoto','FeePerAnimal',
    'sentiment_score','sentiment_magnitude','n_sentences','avg_label_score','n_labels','crop_confidence','desc_length',
    'Breed1_enc','State_enc',
    'rescuer_n_pets','age_rel_breed','word_count','unique_words','avg_word_len','uppercase_ratio','has_exclamation'
]

X_train = train_fe[ALL_FEATURES]; X_test = test_fe[ALL_FEATURES]
y_train = train_fe['AdoptionSpeed']; y_test = test_fe['AdoptionSpeed']
print(f'Features: {len(ALL_FEATURES)}')


Features: 48


## Sección C: LightGBM 5-fold CV

Reentrenamos LightGBM con los mejores hiperparámetros del notebook 05 para obtener
las probabilidades por clase sobre el test set.

In [3]:
lgb_params = {'objective': 'multiclass', 'num_class': 5, 'verbosity': -1,
              'num_leaves': 51, 'lambda_l1': 0.10, 'lambda_l2': 7.58,
              'feature_fraction': 0.59, 'bagging_fraction': 0.98,
              'bagging_freq': 1, 'min_child_samples': 118, 'learning_rate': 0.093}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lgb_test_preds = np.zeros((len(X_test), 5))
lgb_cv = []

for tr_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    m = lgb.train(lgb_params, lgb.Dataset(X_tr, label=y_tr),
                  num_boost_round=500, valid_sets=[lgb.Dataset(X_val, label=y_val)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])
    lgb_cv.append(cohen_kappa_score(y_val, m.predict(X_val).argmax(axis=1), weights='quadratic'))
    lgb_test_preds += m.predict(X_test)

lgb_kappa = cohen_kappa_score(y_test, lgb_test_preds.argmax(axis=1), weights='quadratic')
print(f'LightGBM — CV: {np.mean(lgb_cv):.4f} | Test: {lgb_kappa:.4f}')


LightGBM — CV: 0.3990 | Test: 0.3851


## Sección D: XGBoost 5-fold CV

XGBoost usa una estrategia de crecimiento *depth-wise* (por nivel) en vez de *leaf-wise*.
Esto lo hace más robusto frente a overfitting en datasets pequeños, pero más lento.
La combinación de ambos modelos aprovecha que cometen errores en casos distintos.

In [4]:
xgb_params = {'objective': 'multi:softprob', 'num_class': 5, 'verbosity': 0,
              'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 500,
              'subsample': 0.8, 'colsample_bytree': 0.8,
              'reg_alpha': 0.1, 'reg_lambda': 1.0,
              'min_child_weight': 5, 'random_state': SEED,
              'tree_method': 'hist', 'device': 'cpu'}

xgb_test_preds = np.zeros((len(X_test), 5))
xgb_cv = []

for tr_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              verbose=False)
    probs = model.predict_proba(X_val)
    xgb_cv.append(cohen_kappa_score(y_val, probs.argmax(axis=1), weights='quadratic'))
    xgb_test_preds += model.predict_proba(X_test)

xgb_kappa = cohen_kappa_score(y_test, xgb_test_preds.argmax(axis=1), weights='quadratic')
print(f'XGBoost — CV: {np.mean(xgb_cv):.4f} | Test: {xgb_kappa:.4f}')


XGBoost — CV: 0.4087 | Test: 0.3803


## Sección E: Blend LGB + XGB

Promediamos las probabilidades de ambos modelos con distintos pesos.
El blend es efectivo cuando los modelos tienen correlación baja entre sus errores.

In [5]:
lgb_probs = lgb_test_preds / lgb_test_preds.sum(axis=1, keepdims=True)
xgb_probs = xgb_test_preds / xgb_test_preds.sum(axis=1, keepdims=True)

results = []
for w_lgb in np.arange(0.5, 1.0, 0.05):
    w_xgb = 1 - w_lgb
    blend = w_lgb * lgb_probs + w_xgb * xgb_probs
    kappa = cohen_kappa_score(y_test, blend.argmax(axis=1), weights='quadratic')
    results.append({'w_lgb': round(w_lgb, 2), 'w_xgb': round(w_xgb, 2), 'kappa': round(kappa, 4)})

results_df = pd.DataFrame(results).sort_values('kappa', ascending=False)
best = results_df.iloc[0]
print(results_df.to_string(index=False))

print('='*65)
print('  COMPARATIVA FINAL')
print('='*65)
print(f'  FE v3 + Optuna simple          Test: 0.3595')
print(f'  FE v4 + LightGBM CV            Test: {lgb_kappa:.4f}')
print(f'  FE v4 + XGBoost CV             Test: {xgb_kappa:.4f}')
print(f'  Blend LGB({best["w_lgb"]})+XGB({best["w_xgb"]})   Test: {best["kappa"]:.4f}  <- MEJOR ENSEMBLE')
print('='*65)


 w_lgb  w_xgb  kappa
  0.50   0.50 0.3906
  0.55   0.45 0.3902
  0.60   0.40 0.3891
  0.80   0.20 0.3883
  0.65   0.35 0.3882
  0.90   0.10 0.3871
  0.95   0.05 0.3860
  0.85   0.15 0.3849
  0.70   0.30 0.3838
  0.75   0.25 0.3833
  COMPARATIVA FINAL
  FE v3 + Optuna simple          Test: 0.3595
  FE v4 + LightGBM CV            Test: 0.3851
  FE v4 + XGBoost CV             Test: 0.3803
  Blend LGB(0.5)+XGB(0.5)   Test: 0.3906  <- MEJOR ENSEMBLE


## Seccion F: Optuna para XGBoost

Los parametros de XGBoost en la Seccion D son buenos defaults pero no estan optimizados.
Usamos Optuna con 5-fold CV (igual que en notebook 05 para LightGBM) para encontrar
los mejores hiperparametros. El espacio de busqueda cubre los parametros mas influyentes:
`max_depth`, `learning_rate`, `n_estimators`, `subsample`, `colsample_bytree`,
`reg_alpha`, `reg_lambda`, `min_child_weight`, `gamma`.

⏳ *Esta seccion tarda ~40-60 minutos con 50 trials.*

In [6]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def xgb_cv_objective(trial):
    params = {
        "objective":        "multi:softprob",
        "num_class":        5,
        "verbosity":        0,
        "tree_method":      "hist",
        "device":           "cpu",
        "random_state":     SEED,
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1000),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores = []
    xgb_preds = np.zeros((len(X_test), 5))

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        probs = model.predict_proba(X_val)
        cv_scores.append(cohen_kappa_score(y_val, probs.argmax(axis=1), weights="quadratic"))
        xgb_preds += model.predict_proba(X_test)

    test_kappa = cohen_kappa_score(y_test, xgb_preds.argmax(axis=1), weights="quadratic")
    trial.set_user_attr("test_score", test_kappa)
    return np.mean(cv_scores)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_cv_objective, n_trials=50, show_progress_bar=True)

best_xgb_cv   = study_xgb.best_value
best_xgb_test = study_xgb.best_trial.user_attrs["test_score"]
best_xgb_params = study_xgb.best_params.copy()
best_xgb_params.update({"objective": "multi:softprob", "num_class": 5,
                         "verbosity": 0, "tree_method": "hist",
                         "device": "cpu", "random_state": SEED})

print("="*60)
print(f"Mejor XGB CV (5-fold):  {best_xgb_cv:.4f}")
print(f"Mejor XGB Test:         {best_xgb_test:.4f}")
print(f"Params: {study_xgb.best_params}")
print("="*60)


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor XGB CV (5-fold):  0.4053
Mejor XGB Test:         0.3789
Params: {'n_estimators': 548, 'max_depth': 4, 'learning_rate': 0.046625469541144486, 'subsample': 0.5526685582554334, 'colsample_bytree': 0.7931668765089911, 'reg_alpha': 0.009544579183038799, 'reg_lambda': 0.530280911162103, 'min_child_weight': 9, 'gamma': 1.260811440833841}


## Seccion G: Blend final LGB + XGB optimizado

Reentrenamos XGBoost con los mejores parametros encontrados por Optuna
y combinamos con LightGBM para obtener el mejor ensemble posible.
Comparamos el resultado contra el blend con defaults de la Seccion E.

In [7]:
# Reentrenar XGB con mejores params de Optuna
xgb_opt_preds = np.zeros((len(X_test), 5))
xgb_opt_cv    = []

for tr_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(**best_xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    probs = model.predict_proba(X_val)
    xgb_opt_cv.append(cohen_kappa_score(y_val, probs.argmax(axis=1), weights="quadratic"))
    xgb_opt_preds += model.predict_proba(X_test)

xgb_opt_kappa = cohen_kappa_score(y_test, xgb_opt_preds.argmax(axis=1), weights="quadratic")
print(f"XGBoost optimizado — CV: {np.mean(xgb_opt_cv):.4f} | Test: {xgb_opt_kappa:.4f}")

# Blend LGB + XGB optimizado
lgb_probs_norm = lgb_test_preds / lgb_test_preds.sum(axis=1, keepdims=True)
xgb_probs_norm = xgb_opt_preds  / xgb_opt_preds.sum(axis=1, keepdims=True)

results = []
for w_lgb in np.arange(0.3, 1.0, 0.05):
    w_xgb = round(1 - w_lgb, 2)
    blend = round(w_lgb, 2) * lgb_probs_norm + w_xgb * xgb_probs_norm
    kappa = cohen_kappa_score(y_test, blend.argmax(axis=1), weights="quadratic")
    results.append({"w_lgb": round(w_lgb, 2), "w_xgb": w_xgb, "kappa": round(kappa, 4)})

results_df = pd.DataFrame(results).sort_values("kappa", ascending=False)
best = results_df.iloc[0]
print(results_df.to_string(index=False))

print("="*70)
print("  COMPARATIVA FINAL — Todos los ensembles")
print("="*70)
print(f"  FE v4 + LightGBM CV (nb05)          Test: {lgb_kappa:.4f}")
print(f"  FE v4 + XGB defaults (Sec D)        Test: {xgb_kappa:.4f}")
print(f"  FE v4 + XGB optimizado (Sec F)      Test: {xgb_opt_kappa:.4f}")
print(f"  Blend LGB+XGB defaults 50/50        Test: 0.3906")
print(f"  Blend LGB({best['w_lgb']})+XGB({best['w_xgb']}) optimizado  Test: {best['kappa']:.4f}  <- MEJOR")
print("="*70)


XGBoost optimizado — CV: 0.4053 | Test: 0.3789
 w_lgb  w_xgb  kappa
  0.35   0.65 0.3871
  0.30   0.70 0.3869
  0.55   0.45 0.3856
  0.60   0.40 0.3854
  0.50   0.50 0.3851
  0.65   0.35 0.3851
  0.40   0.60 0.3847
  0.45   0.55 0.3846
  0.75   0.25 0.3843
  0.95   0.05 0.3836
  0.90   0.10 0.3834
  0.70   0.30 0.3822
  0.80   0.20 0.3821
  0.85   0.15 0.3805
  COMPARATIVA FINAL — Todos los ensembles
  FE v4 + LightGBM CV (nb05)          Test: 0.3851
  FE v4 + XGB defaults (Sec D)        Test: 0.3803
  FE v4 + XGB optimizado (Sec F)      Test: 0.3789
  Blend LGB+XGB defaults 50/50        Test: 0.3906
  Blend LGB(0.35)+XGB(0.65) optimizado  Test: 0.3871  <- MEJOR
